# Day 13 — Delta Lake Change Data Feed (CDF) & Incremental Processing


CDF allows downstream consumers to process changes instead of repeatedly comparing entire table snapshots.

**CDF metadata columns**
```
**Column**	        **Purpose**
_change_type	    Type of change
_commit_version	    Delta table version where the change occurred
_commit_timestamp	Timestamp of the Delta commit

```

The _change_type column can contain:
```
**Value**	        **Meaning**
insert	            New record inserted
delete	            Existing record deleted
update_preimage	    Record before an update
update_postimage	Record after an update

```
**_NOTE_**: CDF captures changes only after it has been enabled. It is not a historical change log for updates that happened before activation.


# Part 2
## Step 1 — Create the source Delta table

In [0]:
from pyspark.sql import functions as F

data = [
    (1001, "Amit", "Active", 500.0),
    (1002, "Rahul", "Active", 800.0),
    (1003, "Priya", "Active", 300.0)
]

columns = ["custId", "customerName", "status", "balance"]

df = spark.createDataFrame(data, columns)

source_table = ("workspace.customer360_bronze.customers_cdf_day13")
df.write.format("delta").mode("overwrite").saveAsTable(source_table)


## Verify the Table-

In [0]:
%sql
SELECT * FROM workspace.customer360_bronze.customers_cdf_day13;

## Step 2 — Enable Change Data Feed
** CDF captures changes only after it has been enabled. It is not a historical change log for updates that happened before activation.

In [0]:
%sql
ALTER TABLE workspace.customer360_bronze.customers_cdf_day13
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

## Step 3 — Capture the starting version
Before applying any changes, run:

when table is created --> **version 0 ** \
After enabling the CDF --> **version 1**

In [0]:
%sql
DESCRIBE HISTORY workspace.customer360_bronze.customers_cdf_day13;

## Step 4 — Perform an UPDATE

In [0]:
%sql
UPDATE workspace.customer360_bronze.customers_cdf_day13
SET balance = 650.0
WHERE custId = 1001;

## Step 5 — Perform a DELETE

In [0]:
%sql
DELETE FROM workspace.customer360_bronze.customers_cdf_day13
WHERE custId = 1002;

## Step 6 — Perform an INSERT

In [0]:
%sql
INSERT INTO workspace.customer360_bronze.customers_cdf_day13
VALUES (1004, 'Sneha', 'Active', 900.0);

In [0]:
%sql
SELECT *
FROM workspace.customer360_bronze.customers_cdf_day13
ORDER BY custId;

# Part 3- Read the Change Data Feed

## Step 7 — Query CDF using SQL

For example, if the first data-changing commit after enabling CDF is version 2:

````
6 WRITE
5 OPTIMIZE
4 DELETE
3 OPTIMIZE
2 UPDATE
1 SET TBLPROPERTIES
0 CREATE OR REPLACE TABLE AS SELECT
````

In [0]:
%sql
SELECT
    custId,
    customerName,
    status,
    balance,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes(
    'workspace.customer360_bronze.customers_cdf_day13',
    2
)
ORDER BY _commit_version, custId;

# Part 4: Read CDF using PySpark

## Step 8 — Read changes in batch mode

In [0]:
source_table = "workspace.customer360_bronze.customers_cdf_day13"

dfCDF = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 2)
    .table(source_table)
)

display(
    dfCDF.select(
        "custId",
        "customerName",
        "balance",
        "_change_type",
        "_commit_version",
        "_commit_timestamp"
    ).orderBy("_commit_version", "custId")
)

## Step 9 — Read only changes from version 4 onward

**NOTE:**
- **startingVersion** is inclusive, and CDF returns row-level changes rather than every operation recorded in Delta history.

In [0]:
dfIncremental = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 4)
    .table(source_table)
)

display(
    dfIncremental.select(
        "custId",
        "balance",
        "_change_type",
        "_commit_version"
    ).orderBy("_commit_version")
)

# Part 5: Incremental CDF Streaming into Silver

## Step 10 — Create a CDF streaming DataFrame

**NOTE**: spark.readStream can continue processing new CDF commits as they arrive.

enable CDF- 

In [0]:
source_table = "workspace.customer360_bronze.customers_cdf_day13"

dfCDFStream = (
    spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 2)
    .table(source_table)
)

## Step 11 — Write changes to a Silver history table

In [0]:
target_table = "workspace.customer360_silver.customers_cdf_history_day13"

checkpoint_path = ("/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/customers_cdf_history_day13")

query = (
    dfCDFStream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

## Step 12 — Verify Silver history

In [0]:
%sql
SELECT
    custId,
    balance,
    _change_type,
    _commit_version
FROM workspace.customer360_silver.customers_cdf_history_day13
ORDER BY _commit_version, custId;

## Step 13 — Test incremental CDF processing

once INSERTED re-run **STEP 11**

In [0]:
%sql
INSERT INTO workspace.customer360_bronze.customers_cdf_day13
VALUES (1005, 'Aadhya', 'Active', 1200.0);

In [0]:
%sql
SELECT
    custId,
    balance,
    _change_type,
    _commit_version
FROM workspace.customer360_silver.customers_cdf_history_day13
ORDER BY _commit_version, custId;

### CDF identifies changed rows; checkpointing prevents reprocessing previously committed source progress during normal recovery. With a Delta sink, this supports exactly-once processing under the supported guarantees.

# Part 6: Applying CDF changes using MERGE

Your current Silver history table contains:
````
- update_preimage
- update_postimage
- delete
- insert
```

But a current-state Silver customer table should contain only the latest active records, with deletions applied.

We'll implement this using Delta MERGE.



## Step 14 — Create a current-state Silver table

In [0]:
initial_data = [
    (1001, "Amit", "Active", 500.0),
    (1002, "Rahul", "Active", 800.0),
    (1003, "Priya", "Active", 300.0)
]

dfInitial = spark.createDataFrame(
    initial_data,
    ["custId", "customerName", "status", "balance"]
)

target_table = "workspace.customer360_silver.customers_current_day13"

(
    dfInitial.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

## Step 15 — Prepare the CDF records

We don't need update_preimage for maintaining a current-state table.

We need:
- insert → Insert or update the customer.
- update_postimage → Update the customer.
- delete → Delete the customer.
However, we must process changes in commit order. If multiple changes to the same customer occur, directly merging all CDF rows together can produce incorrect results or ambiguous matches.

For this exercise, we'll select the latest change per customer from the history we've already captured.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dfChanges = (
    spark.table(
        "workspace.customer360_silver.customers_cdf_history_day13"
    )
    .filter(
        F.col("_change_type").isin(
            "insert", "update_postimage", "delete"
        )
    )
)

windowSpec = (
    Window.partitionBy("custId")
    .orderBy(F.col("_commit_version").desc())
)

dfLatestChanges = (
    dfChanges
    .withColumn("rn", F.row_number().over(windowSpec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(dfLatestChanges)

# Mini Challenge

## Step 16 — Register the temporary view

In [0]:
dfLatestChanges.createOrReplaceTempView("latest_customer_changes")

## Step 17 — Execute the MERGE

implemented a Delta Lake CDF pipeline that captures changes and applies them to a current-state Silver table using MERGE.

In [0]:
%sql
MERGE INTO workspace.customer360_silver.customers_current_day13 AS target
USING latest_customer_changes AS source
ON target.custId = source.custId

WHEN MATCHED AND source._change_type = 'delete'
THEN DELETE

WHEN MATCHED AND source._change_type IN ('insert', 'update_postimage')
THEN UPDATE SET
    target.customerName = source.customerName,
    target.status = source.status,
    target.balance = source.balance

WHEN NOT MATCHED AND source._change_type IN ('insert', 'update_postimage')
THEN INSERT (
    custId,
    customerName,
    status,
    balance
)
VALUES (
    source.custId,
    source.customerName,
    source.status,
    source.balance
);

## Step 18 — Validate the results

In [0]:
%sql
SELECT *
FROM workspace.customer360_silver.customers_current_day13
ORDER BY custId;

Important interview concept
Why do we filter out update_preimage?

Because update_preimage contains the old values before an UPDATE. For an SCD Type 1 current-state table, we want the latest values from update_postimage.

Also, notice that our MERGE handles three scenarios:
- MATCHED + DELETE: Removes an existing customer.
- MATCHED + INSERT/UPDATE: Updates an existing customer.
- NOT MATCHED + INSERT/UPDATE: Inserts a new customer.
A DELETE for a customer that does not exist in the target is ignored.

**Q1. What is the difference between Delta Change Data Feed and CDC?**

CDF is a Delta Lake feature that exposes row-level changes made to a Delta table. CDC is a broader approach for capturing and propagating changes between systems. CDF can serve as a source for a CDC pipeline.

**Q2. How would you build an incremental pipeline using CDF?**

I would enable CDF on the source Delta table, read changes using Structured Streaming, and maintain a checkpoint to track processing progress. For a current-state Silver table, I would apply inserts, updates, and deletes using Delta MERGE, with appropriate change ordering and idempotent retry handling.

**Q3. What is the difference between CDF and SCD Type 2?**

CDF provides a feed of changes from a Delta table. SCD Type 2 is a data-modeling technique that preserves historical versions of business records. CDF can be used to drive an SCD Type 2 pipeline, but it does not automatically create SCD Type 2 history.


**NOTE**- 
- **Checkpoint**: Tracks streaming processing progress and helps Spark recover after failures.
- **Commit version:** Helps determine whether an incoming change is newer than the change already applied to a customer.